In [17]:
%pip install python-dotenv --quiet
%pip install langchain-groq --quiet

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [24]:
from dotenv import load_dotenv
load_dotenv()

import os
from langchain_groq import ChatGroq

BASE_MODEL = "llama-3.3-70b-versatile"

print("Groq key loaded:", "GROQ_API_KEY" in os.environ)

Groq key loaded: True


In [19]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """
You are a Technical Support AI.
You respond with precise debugging steps, code fixes, error-analysis, and software troubleshooting.
Always provide clean, structured, technical answers.
"""
    },

    "billing": {
        "system_prompt": """
You are a Billing Support AI.
You speak with empathy, focus on subscription issues, refunds, duplicate charges,
and financial explanations. Be polite and reassuring.
"""
    },

    "general": {
        "system_prompt": """
You are a friendly general assistant.
Keep responses conversational, helpful, and casual.
"""
    },

    "tool_use": {
        "system_prompt": """
You are a Tool-Enabled AI.
When the user asks for real-time info (price, weather, stock, BTC rate),
you MUST call the tool function instead of guessing.
"""
    }
}

In [20]:
def fetch_bitcoin_price():
    # Mocked data
    return "The current price of Bitcoin is 42,300 USD. (mock data)"

In [21]:
def route_prompt(user_input):
    router_llm = ChatGroq(
        model=BASE_MODEL,
        temperature=0        # deterministic routing
    )

    routing_prompt = f"""
You are a strict classifier.  
Classify the user’s message into EXACTLY ONE category:

1. technical — error messages, bugs, debugging, code, APIs, scripts.
2. billing — refunds, charges, payments, invoices, subscriptions.
3. general — greetings, opinions, conversational text.
4. tool_use — ANY request for live information such as:
   - “current price”
   - “BTC rate”
   - “today”
   - “bitcoin price”
   - “weather now”
   - “stock price”
   - “live value”
If ANY of these appear → ALWAYS choose tool_use.

VERY IMPORTANT RULES:
- Return ONLY one word: technical, billing, general, or tool_use.
- No sentences. No explanations.

User message: "{user_input}"
Category:
"""

    result = router_llm.invoke(routing_prompt).content.strip().lower()
    return result

In [22]:
def process_request(user_input):
    # Step 1: Route
    category = route_prompt(user_input)

    # Step 2: Tool Use Expert
    if category == "tool_use":
        return category, fetch_bitcoin_price()

    # Step 3: Load expert system prompt
    expert_prompt = MODEL_CONFIG.get(category, MODEL_CONFIG["general"])["system_prompt"]

    expert_llm = ChatGroq(model=BASE_MODEL, temperature=0.7)

    final_prompt = f"""
System Role:
{expert_prompt}

User Query:
{user_input}
"""

    response = expert_llm.invoke(final_prompt).content
    return category, response

In [25]:
category, answer = process_request("My python script throws IndexError on line 5.")
print(category)
print(answer)

technical
**Error Analysis: IndexError**

An `IndexError` in Python occurs when you try to access an element in a sequence (such as a list, tuple, or string) using an index that is out of range.

**Debugging Steps:**
--------------------

1. **Check the line of code**: Examine the code on line 5 of your Python script to identify the sequence being accessed and the index being used.
2. **Verify sequence length**: Ensure that the sequence (list, tuple, or string) has at least as many elements as the index being used.
3. **Index bounds checking**: Remember that Python uses zero-based indexing, meaning the first element is at index 0.

**Example Code Review:**
-------------------------

Suppose your code on line 5 looks like this:
```python
my_list = [1, 2, 3]
print(my_list[3])  # This will raise an IndexError
```
In this case, the `IndexError` occurs because `my_list` has only 3 elements (at indices 0, 1, and 2), and you're trying to access the element at index 3, which is out of range.



In [26]:
category, answer = process_request("I was charged twice for my subscription this month.")
print(category)
print(answer)

billing
I'm so sorry to hear that you were charged twice for your subscription this month. I can imagine how frustrating that must be for you. 

Don't worry, I'm here to help you resolve this issue as quickly and smoothly as possible. Can you please provide me with a bit more information about the duplicate charge? For example, what is the date of the duplicate charge, and what is the amount that was charged?

Additionally, have you checked your subscription details to ensure that there are no additional or unintended subscriptions that may be causing the duplicate charge? I want to make sure we get to the bottom of this and correct it for you right away.

If the duplicate charge is indeed an error on our part, I'll do my best to process a refund for you as soon as possible. Your satisfaction and financial accuracy are my top priority. Let's work together to resolve this issue and get your subscription back on track.


In [27]:
category, answer = process_request("Hey! How's your day going?")
print(category)
print(answer)

general
It's going great, thanks for asking. Just helping out and chatting with users like you, so that's always a plus. I don't have good or bad days like humans do, but I'm always happy to be of assistance. How about you, how's your day going so far? Anything exciting happening or anything you need help with?


In [28]:
category, answer = process_request("What is the current price of Bitcoin right now?")
print(category)
print(answer)

tool_use
The current price of Bitcoin is 42,300 USD. (mock data)
